<h1>🧭 Biofilter — Report: <code>annotate_go</code></h1>

Everything the bundle knows about a list of Gene Ontology terms: id, name and namespace, where the term sits in the ontology, and what else in the bundle is linked to it.

### 1. Open a bundle

In [ ]:
from biofilter import Biofilter

# A bundle is a directory — the one holding manifest.json.
# Leave as None to use `[database] bundle` from .biofilter.toml.
BUNDLE = None
REPORT = "annotate_go"

bf = Biofilter(bundle=BUNDLE, debug_mode=False) if BUNDLE else Biofilter(debug_mode=False)
bf

### 2. What the report offers

In [ ]:
print("columns:")
for column in bf.report.available_columns(REPORT):
    print(" ", column)

print("\nexample input:")
print(bf.report.example_input(REPORT))

In [ ]:
print(bf.report.explain(REPORT))

### 3. Run it

⚠️ **Terms resolve by id, not by name.** The bundle carries GO codes as
aliases but not term names, so `GO:0006915` resolves and
`apoptotic process` does not. That matches the report this replaces;
changing it would be an ETL change, not a report one.

In [ ]:
terms = [
    "GO:0006915",    # apoptotic process
    "GO:0008150",    # biological_process, near the root
    "GO:9999999",    # kept, with status='not_found'
]

result = bf.report.run(REPORT, input_data=terms)
df = result.to_pandas()
df[["input_value", "go_id", "go_name", "go_namespace", "status"]]

### 4. Where the term sits in the ontology

`go_parent_count` and `go_child_count` are ontology edges. A term near the
root has many children and few parents; a leaf is the reverse.

In [ ]:
df[[
    "go_id",
    "go_name",
    "go_parent_count",
    "go_child_count",
    "go_parent_relation_types",
    "go_child_relation_types",
]]

In [ ]:
def as_list(value):
    """Nullable list column to a Python list. `value or []` raises on an array."""
    return [] if value is None else list(value)


for _, row in df[df["status"] == "ok"].iterrows():
    print(f"{row['go_id']}  {row['go_name']}")
    print("  parents:", as_list(row["go_parent_ids"])[:5])
    print("  children:", as_list(row["go_child_ids"])[:5])
    print()

The id lists are capped by `max_go_terms_per_side` (25 by default), so a
term near the root shows the first 25 children, not all of them. **The
counts are never capped** — trust those.

### 5. Ontology edges are not relationships

`go_parent_count` says where the term sits. `entity_relationships_by_group`
says what else in the bundle is linked to it — genes annotated with the
term, mostly. A term can be deep in the ontology and annotate nothing.

In [ ]:
df[["go_id", "go_child_count", "total_entity_relationships",
    "entity_relationships_by_group"]]

### 6. Every term in the bundle

In [ ]:
import time

started = time.perf_counter()
everything = bf.report.run(REPORT, input_data="__ALL__",
                           include_go_relation_details=False)
catalog = everything.to_pandas()

print(f"{everything.num_rows:,} terms in {time.perf_counter() - started:.1f}s")
catalog.groupby("go_namespace")[["go_parent_count", "go_child_count"]].agg(
    terms="size", mean_children=("go_child_count", "mean")
) if False else catalog["go_namespace"].value_counts()

### 7. Export

CSV by default, with a `.provenance.json` beside it naming the bundle the ids came from.

In [ ]:
for path in result.write("annotate_go.csv"):
    print(path)

### 8. The same thing on the command line

```bash
biofilter report run --report-name annotate_go \\
    --input ... \\
    --output out.csv
```

### 9. Quick QA

In [ ]:
expected = list(bf.report.available_columns(REPORT))
missing = [c for c in expected if c not in df.columns]

print("missing columns:", missing or "none")
print("unresolved inputs:", int((df["status"] == "not_found").sum()))
print("bundle:", result.provenance["bundle_id"])
display(df.dtypes.to_frame("dtype"))